# Spiking wildfire: Colab sweep

Runs the LIF parameter sweep and the LSTM / ConvLSTM baselines on GPU.

**Runtime:** Runtime → Change runtime type → GPU.

This repo is already on Google Drive (`My Drive/spiking-wildfire`). The notebook mounts that folder and writes results into `data/results/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/My Drive/spiking-wildfire'
%cd "$ROOT"
!ls

In [ ]:
!pip -q install -e .
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
from pathlib import Path
needed = [
    Path('data/firms/events.parquet'),
    Path('data/perimeters/spans.parquet'),
    Path('configs/default.yaml'),
]
missing = [str(path) for path in needed if not path.exists()]
if missing:
    raise FileNotFoundError('Missing: ' + ', '.join(missing) + '. Wait for Drive sync, then re-run.')
print('data ready')

## Baselines

Gradient boosting, LSTM, and ConvLSTM on the same 24-hour new-activity question. LSTM and ConvLSTM use the GPU when it is available.

In [ ]:
!python -u scripts/run_baselines.py

## LIF parameter sweep

On GPU the sweep uses dense LIF updates (the grid is only about 40k cells). One sparse-CPU variant is kept for the operation count comparison.

Expect on the order of 30 to 90 minutes for all settings, depending on the Colab GPU.

In [ ]:
import os
os.environ['SPIKING_DEVICE'] = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
!python -u scripts/run_ablations.py

In [ ]:
import json
from pathlib import Path

for name in ('baselines.json', 'ablations.json'):
    path = Path('data/results') / name
    print('===', name, '===')
    print(path.read_text() if path.exists() else 'missing')